In [1]:
import pandas as pd
import numpy as np

#### Training a Neural Network
1. Create a model
2. Choose a loss function
3. Define a dataset
4. Set an optimizer
5. Run a training loop

In [2]:
animals = pd.read_csv("/Users/srisuphachawla/Downloads/zoo.csv.xls")

In [3]:
animals.head()

,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
0,aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
1,antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
2,bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
3,bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
4,boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1


In [4]:
# Animal name not needed - names do not determine classification
features = animals.iloc[:, 1: -1]

# converting into numpy array for easier handling with pytorch
X = features.to_numpy()
print(X)

[[1 0 0 ... 0 0 1]
 [1 0 0 ... 1 0 1]
 [0 0 1 ... 1 0 0]
 ...
 [1 0 0 ... 1 0 1]
 [0 0 1 ... 0 0 0]
 [0 1 1 ... 1 0 0]]


In [5]:
# define target values (ground truth)

target = animals.iloc[: , -1]
y = target.to_numpy()
print(y)

[1 1 4 1 1 1 1 4 4 1 1 2 4 7 7 7 2 1 4 1 2 2 1 2 6 5 5 1 1 1 6 1 1 2 4 1 1
 2 4 6 6 2 6 2 1 1 7 1 1 1 1 6 5 7 1 1 2 2 2 2 4 4 3 1 1 1 1 1 1 1 1 2 7 4
 1 1 3 7 2 2 3 7 4 2 1 7 4 2 6 5 3 3 4 1 1 2 1 6 1 7 2]


In [6]:
import torch
# allows us to store X and y as tensors, easier to manage
from torch.utils.data import TensorDataset

# Instantiate datatset class
dataset = TensorDataset(torch.tensor(X), torch.tensor(y))

# Access an individual sample
input_sample, label_sample = dataset[0]
print("input sample: ", input_sample)
print("label sample: ", label_sample)

input sample:  tensor([1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 4, 0, 0, 1])
label sample:  tensor(1)


In [7]:
from torch.utils.data import DataLoader

# how many samples in each iter, DL models -> large models -> batching helps process large datasets at once
# shuffle randomizes the data order at each epoch, improving model generalization 
# Epoch = one full pass thru the training dataloader
# Generalization : model performs well on unseen data, rather than just memorizing the training set


# create a dataloader
dataloader = DataLoader(dataset, batch_size= 2, shuffle=True)

for batch_inputs, batch_labels in dataloader:
    print('batch_inputs', batch_labels)
    print('batch_labels: ', batch_labels)

batch_inputs tensor([5, 1])
batch_labels:  tensor([5, 1])
batch_inputs tensor([6, 1])
batch_labels:  tensor([6, 1])
batch_inputs tensor([2, 1])
batch_labels:  tensor([2, 1])
batch_inputs tensor([4, 1])
batch_labels:  tensor([4, 1])
batch_inputs tensor([5, 6])
batch_labels:  tensor([5, 6])
batch_inputs tensor([6, 1])
batch_labels:  tensor([6, 1])
batch_inputs tensor([1, 2])
batch_labels:  tensor([1, 2])
batch_inputs tensor([7, 4])
batch_labels:  tensor([7, 4])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([7, 2])
batch_labels:  tensor([7, 2])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([1, 4])
batch_labels:  tensor([1, 4])
batch_inputs tensor([1, 3])
batch_labels:  tensor([1, 3])
batch_inputs tensor([6, 7])
batch_labels:  tensor([6, 7])
batch_inputs tensor([7, 6])
batch_labels:  tensor([7, 6])
batch_inputs tensor([1, 5])
batch_labels:  tensor([1, 5])
batch_inputs t

### Run a Training Loop
- calculate loss (forward pass)
- compute gradients (backpropagation)
- update model params

 

In [8]:
import torch.nn as nn
import torch.optim as optim

def mean_squared_loss(prediction, target):
    return np.mean((prediction - target) ** 2)

criterion = nn.MSELoss()
loss = criterion(prediction, target)

NameError: name 'prediction' is not defined

In [9]:
# create the dataset and dataloader

# convert pandas into numpy first 
X = torch.tensor(features.to_numpy()).float()
y = torch.tensor(target.to_numpy()).float().unsqueeze(1)

# organize into right data types 
dataset = TensorDataset(X,y)

# load into dataloader to enable batching, batch_size is customizable
dataloader = DataLoader(dataset, batch_size = 4, shuffle = True)

# create the model 
model = nn.Sequential(nn.Linear(16, 2),
                      nn.Linear(2,1))

# Create the loss and optimizer, default lr of 0.001 for DL problems
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001)


In [10]:
num_epochs = 100
 
for epoch in range(num_epochs):
    for data in dataloader:
        # set the gradients to 0
        optimizer.zero_grad()

        # get features and target from the data loader
        features, target = data

        # run a forward pass
        pred = model(features)

        loss = criterion(pred, target)
        loss.backward()

        # update params
        optimizer.step()


#### Limitations of Sigmoid Function
- outputs bounded between 0 and 1
- usable anywhere in a network but the gradients are very small for large and small valuses of x
- cause saturation, leading to the vanishing gradients problem (each gradient depends on the previous one, small gradient -> fail to update the next one properly)

--> Softmax also suffers through the same, 

##### Hence, these two activation functions are not ideal for hidden layers and best for last layer only

#### Acitivation for Hidden or linear layer: RELU
- Rectifies Linear Unit 
- for positive inputs, output equals input
- for negative inputs, output is 0 
- helps overcome vanishing gradients
- nn.RelU()


#### Leakuy ReLu
- Positive - behaves like Relu
- Negative - scaled by a small coefficient (default = 0.01)
- graidnets for negative inputs are non-zero

- nn.LeakyReLu(negative_slope = 0.05)

In [11]:
# eg: 

# Create a ReLU function with PyTorch
relu_pytorch = nn.ReLU()

x_pos = torch.tensor(2.0)
x_neg = torch.tensor(-3.0)

# Apply the ReLU function to the tensors
output_pos = relu_pytorch(x_pos)
output_neg = relu_pytorch(x_neg)

print("ReLU applied to positive value:", output_pos)
print("ReLU applied to negative value:", output_neg)


ReLU applied to positive value: tensor(2.)
ReLU applied to negative value: tensor(0.)


#### Learning Rate and Momentum
- training a neural network - solving an optimization problem ( SGD optimizer)

sgd = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.95)

Two arguments
- learning rate: controls the step size
- momentum: adds inertia to avoid getting stuck


- step size decrease near zero as the gradients get smaller with optimal learning rate
- small learning rate, optimzer takes longer to find the minimum
- big learnin rate - optimer bounces back and fortch and loses sight of the minimum

#### Without Momentum 
- lr = 0.01, momentum 0 (optimizer gets stuck at the first dip of the function)

- momentum = 0.9, --> good 

#### Layer Initialization
- a layer weihts are initialized to small values

In [ ]:
layer = nn.Linear(64, 128)
print(layer.weight.min(), layer.weight.max())
# the output is a weightum sum of inputts from the previous layer
# keepin g both the inpur data and layer weights small ensures stable outputs


tensor(-0.1249, grad_fn=<MinBackward1>) tensor(0.1250, grad_fn=<MaxBackward1>)


In [ ]:
# use uniform initialization for layer0 and layer1 weights
layer = nn.Linear(64, 128)
nn.init.uniform_(layer.weight)

print(layer.weight.min(), layer.weight.max())
# weights value now range from 0 to 1

tensor(8.0109e-05, grad_fn=<MinBackward1>) tensor(0.9999, grad_fn=<MaxBackward1>)


#### Transfer Learnin
- takes a model that was trained on a first task and reuses for a second similar task
eg: traininf a model on US salaries dataset, now we have new dataset based on UK dataset, reusue weights to train on new on 

In [ ]:
layer == nn.Linear(64, 128)
# torch.save(layer, 'layer.pth')

# new_layer = torch.load('layer.pth')

#### Fine Tuning
- a type of transfer learning
- load weights from previous model but with a smaller learning rate
- train part of the network (freeze some of them)
- freeze early layers of network and fine tune layers closer to output layer


In [ ]:
model = nn.Sequential(nn.Linear(64, 128), nn.Linear(128, 256))

for name, param in model.named_parameters():
    # check for first layer's weight
    if name == '0.weight' :
        # Freeze the weight
        param.requires_grad = False

    # Check for second layer's weight
    if name == '1.weight':

        # Freeze this weight
        param.requires_grad = False



### Evaluation of Models
- training : adjust model params
- validation : tunes hyperparams
- test : evaluate final model performance


----> calculate training Loss

for each epoch, sum the loss across all batches in the dataloader
- compute the mean training loss at the end of the epoch

In [ ]:
training_loss  = 0.0

for inputs, labels in trainloader:
    # run the forward pass
    outputs = model(inputs)

    # compute the loss
    loss = criterion(outputs, labels)

    # backpropagation 
    loss.backward()   # compute gradients
    optimizer.step()  # update weights
    optimizer.zero_grad()  # reset gradients

    # calculate and sum the loss
    training_loss += loss.item()
epoch_loss = training_loss / len(trainloader)

In [ ]:
# calculating validation loss

validation_loss = 0.0
model.eval() # put model in evaluation mode

with torch.no_grad(): # disable gradients for efficiency
    for inputs, labels in validationloader:
        # run the forward pass
        outputs = model(inputs)

        # calculate the loss
        loss = criterion(outputs, labels)
        validation_loss += loss.item()

epoch_loss = validation_loss / len(validationloader) # compute mean loss
model.train() # switch back to training mode

# overfit model.- training loss decreases, validation loss rises 
# ie the model is learning the trainign data too well and wont perform weel on new data
# doesnt always reflect how accurately it makes predcition

In [ ]:
### calc accuracy woth torchmetrics

import torchmetrics

metric = torchmetrics.Accuracy(task = "multiclass", num_classes = 3)

for features, labels in dataloader:
    outputs: model(features) # forwardpass

    # compute batch accuracy (keeping argmax for one hot labels)
    metric.update(outputs, labels.argmax(dim=-1))
    
# compute accuracy over the whole epoch
accuracy = metric.compute()

# reset for next epoch
metric.reset()

#### fighting overfitting 
- reasons for overfitting 
   - dataset is not large enough ( get more data )
   - model has too much capacity ( reduce model size, add dropout)
   - weights are too large (weight decay to force params to remain small)
   

In [ ]:
# regularization using a dropout layer 
# randomly zeroes out elements of the input tensor during training

model = nn.Sequential(nn.Linear(8, 4),
                      nn.ReLU(),
                      nn.Dropout(p=0.5))

features = torch.randn((1, 8))
print(model(features))
# dropout is added after the activation function
# model.train() for training
# model.eval() to disable dropout during evaluation 


tensor([[0.0000, 0.9601, 0.7943, 0.0000]], grad_fn=<MulBackward0>)


In [ ]:
# example: 

model = nn.Sequential(
    nn.Linear(8, 6),
    nn.Linear(6, 4),
    nn.Dropout(p=0.5))

model.train()
output_train = model(features)

# Forward pass in evaluation mode (Dropout disabled)
model.eval()
output_eval = model(features)

# Print results
print("Output in train mode:", output_train)
print("Output in eval mode:", output_eval)

In [ ]:
# Regularization with weight decay
optimizer = optim.SGD(model.parameters(), lr = 0.001, weight_decay = 0.0001)

# controlled by the weight decay param in the optimizer, typically set to a small val  - 0.0001
# weight decay encourages smaller weights by adding a penakuty during optimization
# helps reduce overfitting, keeping weights smalller and imporiving generalization
# higher weight decay -> stronger the regularization (less likely to overfit)

#### Data Augmentation 
- new data is expensive
- we have a way to expand data artificially
- diff views through rotation and scaled

In [ ]:
# modify the training loop to overfit a single data point
 
features, labels = next(iter(dataloader))

for i in range(1000):
    outputs = model(features)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# should reach 1.0 accuracy and 0 loss if nodel is set up correctly

# step 2: reduce overfitting - maximize validation accuracy
# keep track of each hyperparams and validation accuracy


# step 3: fine tune hyperparameters
# grid search

for factor in range(2, 6):
    lr = 10 ** -factor

# random search - typically more efficient as it avoids unnecesary tests and increases the chance of finding optimal setting
factor = np.random.uniform(2, 6)
lr = 10 ** -factor 





### summary

Chapter 1:
- created small neural networks  
- linear layers

Chapter 2:
- used loss and activation fucntions
- calc derivatives
- backpropagation

Chapter 3:
- trained a neural network
- learning rate and momentum
- impact of the above

Chapter 4:
- strategies to improve yoru model
- reduced overfitting
- evaluated model performance


